# Pipeline de Avaliação de Modelos de Detecção de Anomalias — Censo Agropecuário

Este notebook implementa um pipeline completo de detecção de anomalias para os dados do Censo Agropecuário 2017, utilizando modelos da biblioteca **PyOD**. O pipeline foi desenhado para dados não-paramétricos e esparsos, com taxa de contaminação estimada em **6,87%**.

## Fluxo geral

```
Dados brutos (parquet)
       │
       ▼
 Merge com ground truth (KEY_COLS)
       │
       ▼
 Amostragem estratificada (10k)
       │
       ▼
 Split treino/teste (80/20)
       │
       ▼
 Codificação (AnomalyDetectionEncoder)
       │
       ▼
 Imputação residual (SimpleImputer fill=0)
       │
       ├─── Experimento redução de dimensionalidade (Seção 4b)
       │
       ▼
 Treinamento e scoring (MODELS_REGISTRY)
       │
       ▼
 Métricas por modelo/dataset (ROC-AUC, AP, P@K, F1)
       │
       ▼
 Análise de concordância entre modelos
       │
       ▼
 Avaliação combinada multi-dataset (max score)
       │
       ▼
 Explicabilidade (SHAP — melhor modelo)
       │
       ▼
 Exportação de resultados
```

### Colab

In [ ]:
# !pip install pyod combo

In [1]:
from google.colab import runtime

runtime.unassign()

## Seção 1 — Setup e Configuração

In [ ]:
import sys
import os
import warnings
import time
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

from pyod.models.iforest import IForest
from pyod.models.hbos import HBOS
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.loda import LODA
from pyod.models.lof import LOF
from pyod.models.cblof import CBLOF
from pyod.models.auto_encoder import AutoEncoder
from pyod.models.inne import INNE

# Modelos que possivelmente não serão utilizados
from pyod.models.kpca import KPCA
from pyod.models.feature_bagging import FeatureBagging
from pyod.models.abod import ABOD
from pyod.models.sod import SOD
from pyod.models.rod import ROD
from pyod.models.lscp import LSCP

import torch

# Detectar GPU para AutoEncoder
_ae_device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Dispositivo para AutoEncoder: {_ae_device}")

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

PATH_PREFIX = '../data'
# PATH_PREFIX = '.'
KEY_COLS        = ['V010100', 'NUM_QUADRA', 'NUM_FACE', 'V010800']
CONTAMINATION   = 0.0687
SAMPLE_SIZE     = 10_000
RANDOM_STATE    = 42
LABELS_PATH     = f'{PATH_PREFIX}/experimentos/ground_truth.csv'
DATA_DIR        = f'{PATH_PREFIX}/experimentos/abordagem1'
OUTPUT_DIR      = f'{PATH_PREFIX}/experimentos/abordagem1'

print("Setup concluído.")
print(f"  CONTAMINATION = {CONTAMINATION}")
print(f"  SAMPLE_SIZE   = {SAMPLE_SIZE}")
print(f"  LABELS_PATH   = {LABELS_PATH}")

### Anomaly Encoder

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer


class AnomalyDetectionEncoder(BaseEstimator, TransformerMixin):
    """
    Encoder customizado para detecção de anomalias em dados do Censo Agropecuário.
    
    Estratégia:
    - Variáveis categóricas nominais: Frequency Encoding (frequência normalizada)
    - Variáveis binárias (0/1): Sem transformação
    - Variáveis discretas de baixa cardinalidade (2-10 valores): Sem transformação
    - Variáveis discretas de média cardinalidade (10-50 valores): RobustScaler
    - Variáveis contínuas: RobustScaler (preserva estrutura de outliers)
    
    Parâmetros
    ----------
    key_cols : list, opcional (default=None)
        Lista de colunas de chave que não devem ser transformadas.
    
    Atributos
    ---------
    column_types_ : dict
        Dicionário mapeando tipos de variáveis para listas de nomes de colunas.
        
    transformer_ : ColumnTransformer
        Transformer sklearn configurado com as estratégias apropriadas.
        
    feature_names_out_ : list
        Lista ordenada de nomes de features após transformação.
    
    Nota
    ----
    PowerTransformer foi removido pois pode "normalizar" anomalias, dificultando
    sua detecção. RobustScaler preserva a estrutura de outliers.
    
    Exemplos
    --------
    >>> from anomaly_encoder import AnomalyDetectionEncoder
    >>> 
    >>> # Criar encoder
    >>> encoder = AnomalyDetectionEncoder(key_cols=['ID', 'MUNICIPIO'])
    >>> 
    >>> # Fit apenas nos dados de treino
    >>> encoder.fit(X_train)
    >>> 
    >>> # Transform em treino e teste
    >>> X_train_encoded = encoder.transform(X_train)
    >>> X_test_encoded = encoder.transform(X_test)
    >>> 
    >>> # Visualizar transformações aplicadas
    >>> info = encoder.get_feature_info()
    >>> print(info.groupby('transformation').size())
    """
    
    def __init__(self, key_cols=None):
        self.key_cols = key_cols or []
        self.category_maps_ = {}  # Para armazenar frequency encodings
    
    def _preprocess_dataframe(self, X):
        """
        Pré-processa o dataframe para lidar com tipos especiais.
        
        Parâmetros
        ----------
        X : pd.DataFrame ou array-like
            Dados de entrada.
            
        Retorna
        -------
        pd.DataFrame
            DataFrame processado (mantém categorias para encoding posterior).
        """
        X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
        
        # Substituir valores infinitos por NaN em colunas numéricas
        for col in X_df.columns:
            if X_df[col].dtype.name not in ['category', 'object'] and not str(X_df[col].dtype).startswith('string'):
                X_df[col] = X_df[col].replace([np.inf, -np.inf], np.nan)
        
        return X_df
        
    def _categorize_columns(self, X):
        """
        Categoriza as colunas por tipo para aplicar transformações adequadas.
        
        Parâmetros
        ----------
        X : pd.DataFrame
            DataFrame de entrada (já pré-processado).
            
        Retorna
        -------
        dict
            Dicionário com listas de colunas por categoria:
            - categorical: Variáveis categóricas nominais (tipo 'category')
            - binary: Variáveis binárias (0/1)
            - discrete_low: Discretas baixa cardinalidade (<10 valores)
            - discrete_med: Discretas média cardinalidade (10-50 valores)
            - continuous: Contínuas (todas usam RobustScaler)
        """
        cols = [c for c in X.columns if c not in self.key_cols]
        
        categorical_cols = []
        binary_cols = []
        discrete_low_card = []
        discrete_med_card = []
        continuous_cols = []
        
        for col in cols:
            dtype = X[col].dtype
            
            # Separar categóricas nominais (tipo 'category')
            if dtype.name == 'category':
                categorical_cols.append(col)
                continue
            
            # Ignorar colunas string/object que não são numéricas
            if dtype == 'object' or str(dtype).startswith('string'):
                try:
                    pd.to_numeric(X[col])
                    continue
                except:
                    continue
            
            # Trabalhar com valores numéricos
            values = X[col]
            n_unique = values.nunique()
            n_total = values.notna().sum()
            
            # Variáveis binárias (0/1)
            unique_vals = set(values.dropna().unique())
            if n_unique == 2 and unique_vals.issubset({0, 1, 0.0, 1.0, -1}):
                binary_cols.append(col)
            
            # Variáveis discretas vs contínuas
            elif n_unique < 10:
                discrete_low_card.append(col)
            
            elif n_unique < 50 or (n_unique / n_total) < 0.01:
                discrete_med_card.append(col)
            
            else:
                # Contínua - usar RobustScaler (preserva estrutura de outliers)
                continuous_cols.append(col)
        
        return {
            'categorical': categorical_cols,
            'binary': binary_cols,
            'discrete_low': discrete_low_card,
            'discrete_med': discrete_med_card,
            'continuous': continuous_cols
        }
    
    def _get_frequency_encoding(self, series, is_fit=True):
        """
        Calcula ou aplica frequency encoding para uma série categórica.
        
        Parâmetros
        ----------
        series : pd.Series
            Série categórica a ser encoded.
        is_fit : bool, opcional (default=True)
            Se True, calcula e armazena o mapeamento de frequências.
            Se False, aplica o mapeamento previamente calculado.
            
        Retorna
        -------
        pd.Series
            Série com valores substituídos por frequências normalizadas.
        """
        col_name = series.name
        
        if is_fit:
            # Calcular frequências normalizadas
            freq_map = series.value_counts(normalize=True, dropna=False).to_dict()
            self.category_maps_[col_name] = freq_map
        
        # Aplicar mapeamento e converter para float (necessário para séries categóricas)
        encoded = series.map(self.category_maps_[col_name])
        return pd.to_numeric(encoded, errors='coerce').fillna(0.0)
    
    def fit(self, X, y=None):
        """
        Aprende os parâmetros de transformação apenas nos dados de treino.
        
        Parâmetros
        ----------
        X : pd.DataFrame ou array-like, shape (n_samples, n_features)
            Dados de treino.
            
        y : ignorado
            Não utilizado, presente por compatibilidade sklearn.
            
        Retorna
        -------
        self : AnomalyDetectionEncoder
            Encoder ajustado.
        """
        X_df = self._preprocess_dataframe(X)
        
        # Categorizar colunas
        self.column_types_ = self._categorize_columns(X_df)
        
        # Criar transformadores
        transformers = []
        
        # 1. Variáveis binárias - sem transformação
        if self.column_types_['binary']:
            transformers.append((
                'binary',
                'passthrough',
                self.column_types_['binary']
            ))
        
        # 2. Variáveis discretas de baixa cardinalidade - sem transformação
        if self.column_types_['discrete_low']:
            transformers.append((
                'discrete_low',
                'passthrough',
                self.column_types_['discrete_low']
            ))
        
        # 3. Variáveis discretas de média cardinalidade - RobustScaler
        if self.column_types_['discrete_med']:
            transformers.append((
                'discrete_med',
                RobustScaler(),
                self.column_types_['discrete_med']
            ))
        
        # 4. Variáveis contínuas - RobustScaler (preserva outliers)
        if self.column_types_['continuous']:
            transformers.append((
                'continuous',
                RobustScaler(),
                self.column_types_['continuous']
            ))
        
        # Criar ColumnTransformer
        self.transformer_ = ColumnTransformer(
            transformers=transformers,
            remainder='drop',
            verbose_feature_names_out=False
        )
        
        # Calcular frequency encoding para variáveis categóricas
        for col in self.column_types_['categorical']:
            _ = self._get_frequency_encoding(X_df[col], is_fit=True)
        
        # Fit do transformer
        self.transformer_.fit(X_df)
        
        # Guardar nomes das colunas na ordem correta
        self.feature_names_out_ = (
            self.column_types_['categorical'] +
            self.column_types_['binary'] +
            self.column_types_['discrete_low'] +
            self.column_types_['discrete_med'] +
            self.column_types_['continuous']
        )
        
        return self
    
    def transform(self, X):
        """
        Aplica as transformações aprendidas aos dados.
        
        Parâmetros
        ----------
        X : pd.DataFrame ou array-like, shape (n_samples, n_features)
            Dados a serem transformados.
            
        Retorna
        -------
        pd.DataFrame, shape (n_samples, n_features_transformed)
            Dados transformados com nomes de features preservados.
        """
        X_df = self._preprocess_dataframe(X)
        
        # Aplicar frequency encoding para variáveis categóricas
        # Construir dicionário primeiro para evitar fragmentação de memória
        categorical_dict = {
            col: self._get_frequency_encoding(X_df[col], is_fit=False)
            for col in self.column_types_['categorical']
        }
        categorical_encoded = pd.DataFrame(categorical_dict, index=X_df.index)
        
        # Aplicar transformação do transformer para as outras variáveis
        X_transformed = self.transformer_.transform(X_df)
        
        # Combinar categóricas encoded com outras transformadas
        other_encoded = pd.DataFrame(
            X_transformed,
            columns=[c for c in self.feature_names_out_ if c not in self.column_types_['categorical']],
            index=X_df.index
        )
        
        # Concatenar na ordem correta
        result = pd.concat([categorical_encoded, other_encoded], axis=1)
        result = result[self.feature_names_out_]  # Garantir ordem correta
        
        return result
    
    def get_feature_info(self):
        """
        Retorna informações sobre como cada feature foi transformada.
        
        Retorna
        -------
        pd.DataFrame
            DataFrame com colunas 'feature', 'type' e 'transformation'.
        """
        info = []
        for col_type, cols in self.column_types_.items():
            for col in cols:
                info.append({
                    'feature': col,
                    'type': col_type,
                    'transformation': self._get_transformation_name(col_type)
                })
        return pd.DataFrame(info)
    
    def _get_transformation_name(self, col_type):
        """Retorna o nome legível da transformação aplicada."""
        mapping = {
            'categorical': 'Frequency Encoding',
            'binary': 'Passthrough (sem transformação)',
            'discrete_low': 'Passthrough (baixa cardinalidade)',
            'discrete_med': 'RobustScaler',
            'continuous': 'RobustScaler'
        }
        return mapping.get(col_type, 'Unknown')

### Seção 0b — Dependências Opcionais e Detectores Customizados

Os modelos **RCF**, **EIF** e **SubspaceIForest** são implementados em `custom_detectors.py` usando wrappers compatíveis com a interface do PyOD (`BaseDetector`).

> **Para adicionar um novo modelo externo ao pipeline:**
> 1. Crie uma subclasse de `BaseDetector` em `custom_detectors.py`
> 2. Implemente `fit(X)` e `decision_function(X)`
> 3. Adicione uma entrada no `MODELS_REGISTRY` (Seção 4)

## Seção 2 — Carregamento dos Labels (Ground Truth)

O arquivo de ground truth deve ser um CSV com as seguintes colunas:
- `V010100`, `NUM_QUADRA`, `NUM_FACE`, `V010800` — chave composta (KEY_COLS)
- `is_anomaly` — 0 ou 1 (obrigatório)
- `R*` — colunas que sinalizam se o estabelecimento caiu em alguma regra de anotação
- `anotacao_direta` — sinaliza se o estabelecimento foi marcado diretamente como anomalo

Estabelecimentos **não** presentes no CSV receberão `is_anomaly = 0`.

In [ ]:
if not os.path.exists(LABELS_PATH):
    raise FileNotFoundError(
        f"\n[ERRO] Arquivo de ground truth não encontrado: {LABELS_PATH}\n"
        f"Forneça um CSV com as colunas: {KEY_COLS + ['is_anomaly']} (+ 'origem_anotacao' opcional).\n"
        f"Atualize LABELS_PATH na Seção 0 antes de continuar."
    )

labels_df = pd.read_csv(LABELS_PATH)

# Validar colunas obrigatórias
required_cols = KEY_COLS + ['is_anomaly']
missing = [c for c in required_cols if c not in labels_df.columns]
if missing:
    raise ValueError(f"[ERRO] Colunas ausentes no arquivo de labels: {missing}")

# Garantir tipos corretos nas KEY_COLS
for col in KEY_COLS:
    labels_df[col] = labels_df[col].astype(str)

labels_df['is_anomaly'] = labels_df['is_anomaly'].astype(int)

print(f"Labels carregados: {len(labels_df):,} registros")
print("\nDistribuição de is_anomaly:")
print(labels_df['is_anomaly'].value_counts().rename({0: 'Normal (0)', 1: 'Anomalia (1)'}).to_string())
print(f"\nTaxa de anomalias: {labels_df['is_anomaly'].mean():.2%}")

if 'origem_anotacao' in labels_df.columns:
    print("\nDistribuição por origem_anotacao:")
    print(labels_df.groupby(['origem_anotacao', 'is_anomaly']).size().to_string())

## Seção 3 — Pré-processamento

Para cada dataset (estabel, lav_temp, pecu):
1. Carrega o parquet pós-feature-engineering
2. Garante que KEY_COLS existam como strings e faz merge com ground truth (left join → NaN → 0)
3. Amostragem estratificada por `is_anomaly` (até 10k registros)
4. Split 80/20 treino/teste estratificado
5. Ajusta `AnomalyDetectionEncoder` **apenas** no X_train (evita data leakage)
6. Aplica `SimpleImputer(fill_value=0)` nos NaN residuais (semanticamente: ausente = não se aplica = 0)

In [ ]:
def preprocessar_dataset(nome: str, arquivo_parquet: str, labels_df: pd.DataFrame) -> dict:
    """
    Carrega, une com labels, amostra, divide e codifica um dataset.

    Retorna um dicionário com:
      X_train, X_test, y_train, y_test, feature_names,
      encoder, imputer, df_test (com KEY_COLS para rastreabilidade)
    """
    print(f"\n{'='*60}")
    print(f"Dataset: {nome}")
    print(f"{'='*60}")

    # 1. Carregar parquet
    df = pd.read_parquet(arquivo_parquet)
    print(f"  Shape original: {df.shape}")

    # 2. Garantir KEY_COLS como string no df principal
    for col in KEY_COLS:
        if col in df.columns:
            df[col] = df[col].astype(str)
        else:
            raise KeyError(f"Coluna '{col}' não encontrada no dataset '{nome}'. "
                           f"Colunas disponíveis: {list(df.columns[:10])} ...")

    # 3. Merge com ground truth (left join → NaN → 0)
    df = df.merge(
        labels_df[KEY_COLS + ['is_anomaly']],
        on=KEY_COLS,
        how='left'
    )
    df['is_anomaly'] = df['is_anomaly'].fillna(0).astype(int)
    n_positivos = df['is_anomaly'].sum()
    print(f"  Após merge — anomalias: {n_positivos:,} ({n_positivos/len(df):.2%})")

    # 4. Amostragem estratificada
    n_amostras = min(SAMPLE_SIZE, len(df))
    if n_amostras < len(df):
        df = df.groupby('is_anomaly', group_keys=False).apply(
            lambda g: g.sample(
                n=max(1, round(n_amostras * len(g) / len(df))),
                random_state=RANDOM_STATE
            )
        ).reset_index(drop=True)
        print(f"  Após amostragem: {len(df):,} registros "
              f"(anomalias: {df['is_anomaly'].sum():,} = {df['is_anomaly'].mean():.2%})")

    # 5. Separar features, labels e chave
    y = df['is_anomaly'].values
    key_df = df[KEY_COLS].copy()
    X = df.drop(columns=KEY_COLS + ['is_anomaly'], errors='ignore')

    # 6. Split treino/teste estratificado 80/20
    X_train, X_test, y_train, y_test, key_train, key_test = train_test_split(
        X, y, key_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y
    )
    print(f"  Treino: {X_train.shape} | Teste: {X_test.shape}")

    # 7. Encoder — ajustado APENAS no treino
    encoder = AnomalyDetectionEncoder(key_cols=[])   # KEY_COLS já removidas
    encoder.fit(X_train)
    X_train_enc = encoder.transform(X_train)
    X_test_enc  = encoder.transform(X_test)

    # 8. Imputação residual de NaN (fill_value=0: ausente = não se aplica)
    imputer = SimpleImputer(strategy='constant', fill_value=0)
    X_train_enc = imputer.fit_transform(X_train_enc)
    X_test_enc  = imputer.transform(X_test_enc)

    feature_names = encoder.get_feature_info()['feature'].tolist() \
        if hasattr(encoder, 'get_feature_info') else list(X_train.columns)

    print(f"  Shape final (treino): {X_train_enc.shape}")
    print(f"  NaN residuais (teste): {np.isnan(X_test_enc).sum()}")

    return {
        'X_train':      X_train_enc,
        'X_test':       X_test_enc,
        'y_train':      y_train,
        'y_test':       pd.Series(y_test, name='is_anomaly'),
        'key_test':     key_test.reset_index(drop=True),
        'feature_names': feature_names,
        'encoder':      encoder,
        'imputer':      imputer,
    }


# ─────────────────────────────────────────────────────────────
# Processar os 3 datasets
# ─────────────────────────────────────────────────────────────
dataset_files = {
    'estabel': os.path.join(DATA_DIR, 'df_estabel_final.parquet'),
    'lav_temp': os.path.join(DATA_DIR, 'df_lav_temp_final.parquet'),
    'pecu':    os.path.join(DATA_DIR, 'df_pecu_final.parquet'),
}

datasets = {}
for nome, arquivo in dataset_files.items():
    if not os.path.exists(arquivo):
        print(f"[AVISO] Arquivo não encontrado, pulando dataset '{nome}': {arquivo}")
        continue
    datasets[nome] = preprocessar_dataset(nome, arquivo, labels_df)

print(f"\n{'='*60}")
print(f"Datasets processados: {list(datasets.keys())}")

## Seção 4 — Registro de Modelos (MODELS_REGISTRY)

Todos os modelos são instanciados via **factories** (lambdas) para garantir que cada execução crie um objeto fresco, sem estado compartilhado entre datasets.

### Modelos nativos do PyOD
| Modelo | Família |
|--------|---------|
| IForest, LODA, FB, INNE, LSCP | Ensemble |
| ECOD, COPOD | Probabilistic |
| LOF, ABOD, SOD | Proximity |
| HBOS | Histograma |
| ROD | Statistical |

### Modelos externos via `custom_detectors.py` ★
| Modelo | Biblioteca | Diferencial |
|--------|-----------|-------------|
| **RCF** | `rrcf` | Score CoDisp; detecta anomalias em subespaços e streaming |
| **EIF** | `eif` | Cortes oblíquos; elimina viés direcional do IForest padrão |
| **SubspaceIF** | PyOD interno | Agrega IForests em subespaços `sqrt(d)`; detecta anomalias localizadas |

### Como adicionar um novo modelo nativo do PyOD

```python
MODELS_REGISTRY["MeuModelo"] = {
    "factory":   lambda: MinhaClasse(contamination=CONTAMINATION, ...),
    "family":    "Ensemble",   # Ensemble | Probabilistic | Proximity | Histograma | Statistical
    "rationale": "Breve justificativa da escolha",
}
```

### Como adicionar um novo modelo externo

```python
# 1. Em custom_detectors.py — criar wrapper herdando BaseDetector:
class MeuDetectorExterno(BaseDetector):
    def fit(self, X, y=None):
        # treinar modelo externo
        self.decision_scores_ = ...   # score por amostra de treino
        self._process_decision_scores()
        return self
    def decision_function(self, X):
        return ...  # score por amostra de teste

# 2. No notebook (Seção 0b) — importar:
from custom_detectors import MeuDetectorExterno

# 3. No MODELS_REGISTRY — registrar:
MODELS_REGISTRY["MeuExt"] = {
    "factory":   lambda: MeuDetectorExterno(contamination=CONTAMINATION, ...),
    "family":    "Ensemble",
    "rationale": "...",
}
```

> **Famílias** influenciam a estratégia de explicabilidade na Seção 9:
> - `Ensemble` + IForest/SubspaceIF → `shap.TreeExplainer`
> - `Probabilistic` (ECOD/COPOD) → atributos internos de score por feature
> - demais (incluindo RCF e EIF) → `shap.KernelExplainer` (amostra de background)

In [ ]:
MODELS_REGISTRY = {
    "IForest": {
        "factory": lambda: IForest(
            contamination=CONTAMINATION,
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "family": "Ensemble",
        "rationale": (
            "Isolation Forest cria partições aleatórias do espaço de features; "
            "anomalias são isoladas com menos divisões. Robusto a dados esparsos e "
            "de alta dimensionalidade, sem assumir distribuição paramétrica."
        ),
    },
    "HBOS": {
        "factory": lambda: HBOS(
            contamination=CONTAMINATION,
            n_bins=50,
        ),
        "family": "Histograma",
        "rationale": (
            "Histogram-Based Outlier Score calcula a densidade de cada feature "
            "independentemente via histogramas. Extremamente rápido e eficiente "
            "em dados mistos com muitas features binárias/discretas."
        ),
    },
    "ECOD": {
        "factory": lambda: ECOD(
            contamination=CONTAMINATION,
            n_jobs=-1
        ),
        "family": "Probabilistic",
        "rationale": (
            "Empirical Cumulative Distribution-based Outlier Detection usa a "
            "distribuição empírica cumulativa de cada feature. Sem parâmetros "
            "críticos para ajuste; eficaz em dados não-paramétricos."
        ),
    },
    "COPOD": {
        "factory": lambda: COPOD(
            contamination=CONTAMINATION,
            n_jobs=-1
        ),
        "family": "Probabilistic",
        "rationale": (
            "Copula-based Outlier Detection usa cópulas empíricas para modelar "
            "dependências entre features. Boa calibração de scores e complementar ao ECOD."
        ),
    },
    "LOF": {
        "factory": lambda: LOF(
            contamination=CONTAMINATION,
            n_neighbors=20,
            n_jobs=-1
        ),
        "family": "Proximity",
        "rationale": (
            "Local Outlier Factor detecta anomalias por comparação de densidades locais. "
            "Captura anomalias contextuais que modelos globais tendem a perder; "
            "complementa os modelos de ensemble."
        ),
    },
        "CBLOF": {
        "factory": lambda: CBLOF(
            contamination=CONTAMINATION,
            n_clusters=8,
            alpha=0.9,
            beta=5,
            use_weights=False,
            random_state=RANDOM_STATE,
        ),
        "family": "Proximity",
        "rationale": (
            "Cluster-Based Local Outlier Factor atribui scores baseados no tamanho do "
            "cluster de pertencimento e na distância ao cluster mais próximo. Pontos em "
            "clusters pequenos ou distantes de clusters grandes são marcados como anomalias."
        ),
    },
    "AutoEncoder": {
        "factory": lambda: AutoEncoder(
            contamination=CONTAMINATION,
            hidden_neuron_list=[64, 32, 32, 64],
            hidden_activation_name='relu',
            epoch_num=50,
            batch_size=32,
            lr=1e-3,
            random_state=RANDOM_STATE,
            device=_ae_device,
        ),
        "family": "Neural",
        "rationale": (
            "AutoEncoder aprende uma representação comprimida dos dados; registros com "
            "erro de reconstrução elevado são considerados anomalias. Captura padrões "
            "não-lineares complexos. Utiliza GPU se disponível para acelerar o treino."
        ),
    },
    "INNE": {
        "factory": lambda: INNE(
            contamination=CONTAMINATION,
            n_estimators=200,
            random_state=RANDOM_STATE,
        ),
        "family": "Ensemble",
        "rationale": (
            "Isolation-based Nearest Neighbor Ensemble usa isolamento por proximidade "
            "ao invés de partições aleatórias. Lida melhor com densidades variáveis e "
            "anomalias em clusters, sendo complementar ao IForest."
        ),
    },
    "LODA": {
        "factory": lambda: LODA(
            contamination=CONTAMINATION,
            n_bins=50,
        ),
        "family": "Ensemble",
        "rationale": (
            "Lightweight Online Detector of Anomalies usa projeções aleatórias esparsas "
            "em histogramas 1D. Especialmente adequado para dados esparsos e alta "
            "dimensionalidade — cada projeção envolve poucas features originais."
        ),
    },
    # "FB": {
    #     "factory": lambda: FeatureBagging(
    #         contamination=CONTAMINATION,
    #         n_estimators=10,
    #         random_state=RANDOM_STATE,
    #         n_jobs=-1,
    #     ),
    #     "family": "Ensemble",
    #     "rationale": (
    #         "Feature Bagging treina múltiplos detectores LOF em subconjuntos aleatórios "
    #         "de features. Eficaz quando anomalias se manifestam em subespaços específicos "
    #         "do espaço de alta dimensionalidade."
    #     ),
    # },
    # "ABOD": {
    #     "factory": lambda: ABOD(
    #         contamination=CONTAMINATION,
    #         n_neighbors=10,
    #         method='fast',
    #     ),
    #     "family": "Proximity",
    #     "rationale": (
    #         "Angle-Based Outlier Detection mede a variância dos ângulos entre triplas de "
    #         "pontos. Pontos anômalos têm menor variância angular. O método 'fast' usa "
    #         "apenas os k vizinhos mais próximos, tornando-o viável em alta dimensão."
    #     ),
    # },
    # "SOD": {
    #     "factory": lambda: SOD(
    #         contamination=CONTAMINATION,
    #         n_neighbors=20,
    #         ref_set=10,
    #         alpha=0.8,
    #     ),
    #     "family": "Proximity",
    #     "rationale": (
    #         "Subspace Outlier Detection avalia cada ponto no subespaço de maior variância "
    #         "de seus vizinhos. Robusto à maldição da dimensionalidade ao focar nas "
    #         "dimensões mais relevantes localmente."
    #     ),
    # },
    # "ROD": { # Alto consumo de memória e tempo
    #     "factory": lambda: ROD(
    #         contamination=CONTAMINATION,
    #     ),
    #     "family": "Statistical",
    #     "rationale": (
    #         "Rotation-based Outlier Detection aplica rotações nos dados para detectar "
    #         "outliers de forma invariante à escala e orientação das features. "
    #         "Não assume distribuição e é eficaz para dados multivariados."
    #     ),
    # },
    # "LSCP": {
    #     "factory": lambda: LSCP(
    #         detector_list=[
    #             LOF(contamination=CONTAMINATION, n_neighbors=20),
    #             IForest(contamination=CONTAMINATION, n_estimators=100, random_state=RANDOM_STATE),
    #             HBOS(contamination=CONTAMINATION),
    #         ],
    #         contamination=CONTAMINATION,
    #         random_state=RANDOM_STATE,
    #     ),
    #     "family": "Ensemble",
    #     "rationale": (
    #         "Locally Selective Combination in Parallel Outlier Ensembles seleciona "
    #         "localmente os melhores detectores base para cada ponto de teste. "
    #         "Combina LOF, IForest e HBOS, aproveitando a complementaridade entre eles."
    #     ),
    # },
    # ── Modelos externos via custom_detectors.py ──────────────────────────────
    # "RCF": {
    #     "factory": lambda: RCFDetector(
    #         contamination=CONTAMINATION,
    #         n_trees=200,
    #         tree_size=256,
    #         random_state=RANDOM_STATE,
    #     ),
    #     "family": "Ensemble",
    #     "rationale": (
    #         "Random Cut Forest (rrcf) usa Collusive Displacement (CoDisp) como score. "
    #         "Detecta anomalias localizadas em subespaços e é naturalmente adaptado a "
    #         "fluxos de dados. CoDisp amplifica pontos que 'perturbam' a estrutura da árvore."
    #     ),
    # },
    # "EIF": {
    #     "factory": lambda: EIFDetector(
    #         contamination=CONTAMINATION,
    #         n_trees=200,
    #         sample_size=256,
    #         extension_level=None,   # None → n_features - 1 (extensão máxima)
    #         random_state=RANDOM_STATE,
    #     ),
    #     "family": "Ensemble",
    #     "rationale": (
    #         "Extended Isolation Forest (eif) substitui cortes axis-aligned por "
    #         "hiperplanos oblíquos aleatórios. Elimina o viés direcional do IForest "
    #         "padrão, produzindo scores mais isotrópicos — especialmente útil quando "
    #         "anomalias não estão alinhadas com os eixos das features."
    #     ),
    # },
    # "SubspaceIF": {
    #     "factory": lambda: SubspaceIForest(
    #         contamination=CONTAMINATION,
    #         n_subspaces=50,
    #         subspace_size=None,     # None → sqrt(n_features)
    #         n_estimators=100,
    #         random_state=RANDOM_STATE,
    #     ),
    #     "family": "Ensemble",
    #     "rationale": (
    #         "Subspace Isolation Forest (variante IForest): treina 50 IForests em "
    #         "subespaços aleatórios de sqrt(n_features) features e agrega pela média "
    #         "normalizada. Detecta anomalias que se manifestam em combinações específicas "
    #         "de poucas variáveis — padrão frequente em dados de formulários do Censo."
    #     ),
    # },
}

print("MODELS_REGISTRY criado com os modelos:")
for name, cfg in MODELS_REGISTRY.items():
    print(f"  [{cfg['family']:15s}] {name}: {cfg['rationale'][:60]}...")


## Seção 4b — Experimento: Redução de Dimensionalidade

Avalia o impacto de aplicar **PCA** como pré-processamento antes do IForest, usando o dataset `estabel` como referência.

**Caveat sobre PCA em dados do censo:**
> O PCA preserva componentes de **alta variância**, mas anomalias em dados do censo podem se manifestar justamente nas features de **baixa variância** (ex.: combinações raras de variáveis binárias). Ao descartar componentes menores, o PCA pode suprimir o sinal de anomalia. Este experimento verifica empiricamente se o PCA ajuda ou prejudica.

**KPCA como detector direto:** O PyOD oferece `KPCA` como modelo de detecção — usa o erro de reconstrução via kernel PCA como score de anomalia, sem os riscos do PCA como pré-processamento.

In [ ]:
# def _avaliar_dimred(nome_exp, X_train, X_test, y_test, contamination=CONTAMINATION):
#     """Treina IForest com pré-processamento opcional e retorna AP e ROC-AUC."""
#     if 'PCA' in nome_exp:
#         pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
#         X_tr  = pca.fit_transform(X_train)
#         X_te  = pca.transform(X_test)
#         n_comp = X_tr.shape[1]
#         print(f"  PCA: {X_train.shape[1]} → {n_comp} componentes (95% variância)")
#     else:
#         X_tr, X_te = X_train, X_test

#     model = IForest(contamination=contamination, n_estimators=100,
#                     random_state=RANDOM_STATE, n_jobs=-1)
#     model.fit(X_tr)
#     scores = model.decision_function(X_te)

#     ap  = average_precision_score(y_test, scores)
#     auc = roc_auc_score(y_test, scores)
#     print(f"  {nome_exp:<30s}  AP={ap:.4f}  ROC-AUC={auc:.4f}")
#     return {'AP': ap, 'ROC-AUC': auc, 'model': model, 'scores': scores}


# def _avaliar_kpca(X_train, X_test, y_test, contamination=CONTAMINATION):
#     """Avalia KPCA como detector direto (PyOD)."""
#     model = KPCA(contamination=contamination)
#     model.fit(X_train)
#     scores = model.decision_function(X_test)
#     ap  = average_precision_score(y_test, scores)
#     auc = roc_auc_score(y_test, scores)
#     print(f"  {'KPCA (detector direto)':<30s}  AP={ap:.4f}  ROC-AUC={auc:.4f}")
#     return {'AP': ap, 'ROC-AUC': auc}


# # ─────────────────────────────────────────────────────────────
# # Executar experimento no dataset 'estabel'
# # ─────────────────────────────────────────────────────────────
# dimred_results = {}

# if 'estabel' in datasets:
#     ds = datasets['estabel']
#     print("Experimento de redução de dimensionalidade — dataset: estabel\n")

#     dimred_results['IForest (sem redução)'] = _avaliar_dimred(
#         'IForest (sem redução)', ds['X_train'], ds['X_test'], ds['y_test'])

#     dimred_results['IForest + PCA (95%)'] = _avaliar_dimred(
#         'IForest + PCA (95%)', ds['X_train'], ds['X_test'], ds['y_test'])

#     dimred_results['KPCA (detector direto)'] = _avaliar_kpca(
#         ds['X_train'], ds['X_test'], ds['y_test'])

#     # Resumo visual
#     dimred_df = pd.DataFrame(dimred_results).T[['AP', 'ROC-AUC']].astype(float)
#     fig, axes = plt.subplots(1, 2, figsize=(10, 4))
#     dimred_df['AP'].plot(kind='bar', ax=axes[0], color=['steelblue','tomato','seagreen'], rot=15)
#     axes[0].set_title('Average Precision')
#     axes[0].set_ylim(0, 1)
#     for bar in axes[0].patches:
#         axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
#                      f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

#     dimred_df['ROC-AUC'].plot(kind='bar', ax=axes[1], color=['steelblue','tomato','seagreen'], rot=15)
#     axes[1].set_title('ROC-AUC')
#     axes[1].set_ylim(0, 1)
#     for bar in axes[1].patches:
#         axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
#                      f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

#     plt.suptitle('Impacto da Redução de Dimensionalidade — IForest (estabel)', fontsize=12)
#     plt.tight_layout()
#     plt.show()

#     # Recomendação automática
#     melhor = dimred_df['AP'].idxmax()
#     print(f"\n→ Recomendação: '{melhor}' apresentou melhor AP ({dimred_df.loc[melhor, 'AP']:.4f}).")
#     if 'PCA' in melhor:
#         print("  O PCA melhorou o desempenho — considere incluir IForest+PCA no MODELS_REGISTRY.")
#     else:
#         print("  O PCA não trouxe ganho — prosseguir sem redução de dimensionalidade é adequado.")
# else:
#     print("[AVISO] Dataset 'estabel' não disponível. Pulando experimento de redução de dimensionalidade.")

## Seção 5 — Treinamento e Scoring

Treina cada modelo do `MODELS_REGISTRY` em cada dataset disponível. Os modelos são instanciados via factory a cada execução para garantir independência entre datasets.

In [ ]:
def treinar_e_avaliar(model_name: str, model, X_train, X_test):
    """
    Treina o modelo e retorna scores, predições, tempo e o modelo ajustado.
    O modelo ajustado é armazenado para uso posterior na explicabilidade.
    """
    t0 = time.time()
    model.fit(X_train)
    scores = model.decision_function(X_test)
    preds  = model.predict(X_test)
    elapsed = time.time() - t0
    return scores, preds, elapsed, model   # retorna o modelo ajustado


# ─────────────────────────────────────────────────────────────
# Loop principal: treino + scoring em todos datasets × modelos
# ─────────────────────────────────────────────────────────────
# Estrutura: all_results[dataset_name][model_name] = {scores, preds, time, fitted_model}
all_results = {}

for ds_name, ds in datasets.items():
    print(f"\n{'─'*55}")
    print(f"Dataset: {ds_name}  (X_test shape: {ds['X_test'].shape})")
    print(f"{'─'*55}")
    all_results[ds_name] = {}

    for model_name, model_cfg in MODELS_REGISTRY.items():
        model = model_cfg['factory']()
        scores, preds, elapsed, fitted = treinar_e_avaliar(
            model_name, model, ds['X_train'], ds['X_test']
        )
        all_results[ds_name][model_name] = {
            'scores':       scores,
            'preds':        preds,
            'time':         elapsed,
            'fitted_model': fitted,
        }
        n_flagged = (preds == 1).sum()
        print(f"  {model_name:<10s}  tempo={elapsed:5.1f}s  "
              f"flagged={n_flagged:4d} ({n_flagged/len(preds):.2%})")

print("\nTreinamento concluído.")

## Seção 6 — Métricas de Avaliação

#### Métricas clássicas
Assumem que todos os `y=0` são normais (negativo verdadeiro).

| Métrica | Descrição |
|---------|-----------|
| **Average Precision (AP)** | Métrica principal — área sob a curva P×R; penaliza FPs antes de TPs |
| **ROC-AUC** | Capacidade discriminativa global |
| **Precision@K** | Precisão nos top-K scores onde K = `⌊N × contamination⌋` |
| **F1@contamination** | F1 usando threshold no percentil `(1 − contamination)` |

#### Métricas PU (Positive-Unlabeled)
> **Cenário:** todos os `y=1` são anomalias confirmadas, mas `y=0` pode conter anomalias não catalogadas pelas regras. Penalizar um modelo por ranquear um `y=0` alto pode ser injusto. Essas métricas avaliam apenas **como os positivos confirmados são posicionados no ranking**, sem depender da qualidade dos negativos.

| Métrica | Descrição | Baseline aleatória |
|---------|-----------|-------------------|
| **Recall@K** | % dos `y=1` recuperados nos top-K (K = n° de positivos) | ≈ K/N |
| **Recall@2K** | Idem com K dobrado — mede cobertura com margem | ≈ 2K/N |
| **Mean Norm. Rank (MNR)** | Posição média normalizada dos `y=1` no ranking: 0=perfeito, 0.5=aleatório | 0.5 |
| **NDCG@K** | Qualidade do ranking ponderando hits pelo log da posição | baixo |

A entrada `dataset=combined` representa a avaliação combinada multi-dataset: o score final de cada estabelecimento é o **máximo** entre os scores dos datasets disponíveis (estabel, lav_temp, pecu).

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.metrics import ndcg_score


def precision_at_k(y_true, scores, k: int) -> float:
    """Precisão nos top-K scores mais altos."""
    top_k_idx = np.argsort(scores)[::-1][:k]
    return float(np.asarray(y_true)[top_k_idx].mean())


# ── Métricas PU (Positive-Unlabeled) ─────────────────────────────────────────
# Premissa: todos os y=1 são anomalias confirmadas, mas y=0 pode conter
# anomalias não catalogadas. As métricas abaixo ignoram a incerteza dos 0s
# e avaliam apenas como os positivos confirmados são posicionados no ranking.

def recall_at_k(y_true, scores, k: int) -> float:
    """
    Recall@K (PU): fração dos positivos confirmados recuperados nos top-K.
    Não penaliza por 0s ranqueados alto — eles podem ser anomalias reais.
    Baseline aleatória ≈ K / N.
    """
    y = np.asarray(y_true)
    n_pos = int(y.sum())
    if n_pos == 0:
        return 0.0
    top_k_idx = np.argsort(scores)[::-1][:k]
    return float(y[top_k_idx].sum() / n_pos)


def mean_normalized_rank(y_true, scores) -> float:
    """
    Posição média normalizada dos positivos confirmados no ranking de scores.
    Valor em [0, 1]: 0 = todos no topo (perfeito), 0.5 = aleatório, 1 = pior caso.
    Não usa os 0s como negativo — apenas posiciona os 1s no ranking geral.
    """
    y = np.asarray(y_true)
    n = len(y)
    # Rank 0-based: 0 = maior score
    order = np.argsort(scores)[::-1]
    ranks = np.empty(n, dtype=float)
    ranks[order] = np.arange(n)
    pos_ranks = ranks[y == 1]
    if len(pos_ranks) == 0:
        return 0.5
    return float(pos_ranks.mean() / (n - 1))   # normaliza para [0, 1]


def ndcg_at_k(y_true, scores, k: int) -> float:
    """
    NDCG@K: qualidade do ranking ponderando hits pelo log da posição.
    Penaliza mais fortemente positivos que aparecem tarde no ranking.
    Usa apenas os y=1 como relevantes (relevância binária).
    """
    y = np.asarray(y_true, dtype=float)
    s = np.asarray(scores, dtype=float)
    return float(ndcg_score(y.reshape(1, -1), s.reshape(1, -1), k=k))

def calcular_metricas(y_true, scores, contamination: float = CONTAMINATION) -> dict:
    """
    Calcula métricas clássicas + métricas PU (Positive-Unlabeled).

    Métricas clássicas (assumem y=0 como negativo verdadeiro):
      - avg_precision, roc_auc, precision_at_k, f1_at_cont

    Métricas PU (avaliam apenas a posição dos positivos confirmados):
      - recall_at_k    : % dos y=1 nos top-K (K = n_positivos da amostra)
      - recall_at_2k   : idem com K dobrado (captura cobertura mais ampla)
      - mean_norm_rank : posição média normalizada dos y=1 [0=perfeito, 0.5=random]
      - ndcg_at_k      : qualidade do ranking (NDCG) nos top-K
    """
    y = np.asarray(y_true)
    n_pos = max(1, int(y.sum()))
    k_cont = max(1, math.floor(len(y) * contamination))

    # ── Clássicas ─────────────────────────────────────────────────
    threshold = np.percentile(scores, (1 - contamination) * 100)
    preds_at_cont = (scores >= threshold).astype(int)
    tp = int(((preds_at_cont == 1) & (y == 1)).sum())
    fp = int(((preds_at_cont == 1) & (y == 0)).sum())
    fn = int(((preds_at_cont == 0) & (y == 1)).sum())
    denom = 2 * tp + fp + fn
    f1 = (2 * tp / denom) if denom > 0 else 0.0

    # ── PU ────────────────────────────────────────────────────────
    k_pu  = n_pos          # K = número real de positivos (cobertura exata)
    k_pu2 = 2 * n_pos      # K duplo (cobertura com margem)

    return {
        # Clássicas
        'avg_precision':  average_precision_score(y, scores),
        'roc_auc':        roc_auc_score(y, scores),
        'precision_at_k': precision_at_k(y, scores, k_cont),
        'f1_at_cont':     f1,
        'k':              k_cont,
        # PU
        'recall_at_k':    recall_at_k(y, scores, k_pu),
        'recall_at_2k':   recall_at_k(y, scores, k_pu2),
        'mean_norm_rank': mean_normalized_rank(y, scores),
        'ndcg_at_k':      ndcg_at_k(y, scores, k_pu),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Calcular métricas para todos os experimentos
# ─────────────────────────────────────────────────────────────────────────────
metrics_rows = []
for ds_name, ds_results in all_results.items():
    y_test = datasets[ds_name]['y_test']
    for model_name, res in ds_results.items():
        m = calcular_metricas(y_test, res['scores'])
        metrics_rows.append({
            'dataset':        ds_name,
            'model':          model_name,
            'family':         MODELS_REGISTRY[model_name]['family'],
            # Clássicas
            'avg_precision':  m['avg_precision'],
            'roc_auc':        m['roc_auc'],
            'precision_at_k': m['precision_at_k'],
            'f1_at_cont':     m['f1_at_cont'],
            # PU
            'recall_at_k':    m['recall_at_k'],
            'recall_at_2k':   m['recall_at_2k'],
            'mean_norm_rank': m['mean_norm_rank'],
            'ndcg_at_k':      m['ndcg_at_k'],
            'time_s':         all_results[ds_name][model_name]['time'],
        })

ranking_df = pd.DataFrame(metrics_rows).sort_values(
    ['dataset', 'avg_precision'], ascending=[True, False]
)

_cols_show = ['dataset', 'model', 'family',
              'avg_precision', 'roc_auc', 'precision_at_k', 'f1_at_cont',
              'recall_at_k', 'recall_at_2k', 'mean_norm_rank', 'ndcg_at_k',
              'time_s']


# ─────────────────────────────────────────────────────────────────────────────
# Avaliação combinada multi-dataset (dataset='combined')
# combined_score = max(score_estabel, score_lav_temp, score_pecu)
# ─────────────────────────────────────────────────────────────────────────────
def _build_score_df(ds_name: str, model_name: str) -> pd.DataFrame:
    """Constrói DataFrame com KEY_COLS + score + y_true para um dado dataset/modelo."""
    ds = datasets[ds_name]
    scores = all_results[ds_name][model_name]['scores']
    df_out = ds['key_test'].copy()
    df_out[f'score_{ds_name}'] = scores
    df_out['y_true'] = ds['y_test'].values
    return df_out

combined_rows = []
combined_dfs = {}   # {model_name: DataFrame com scores combinados}

for model_name in MODELS_REGISTRY.keys():
    parts = []
    for ds_name in all_results:
        if model_name in all_results[ds_name]:
            parts.append(_build_score_df(ds_name, model_name))
    if not parts:
        continue
    merged = parts[0]
    for part in parts[1:]:
        part_no_ytrue = part.drop(columns=['y_true'])
        merged = merged.merge(part_no_ytrue, on=KEY_COLS, how='outer')
    score_cols = [c for c in merged.columns if c.startswith('score_')]
    merged['combined_score'] = merged[score_cols].max(axis=1)
    merged['y_true'] = merged['y_true'].fillna(0).astype(int)
    combined_dfs[model_name] = merged
    valid = merged.dropna(subset=['combined_score'])
    if valid['y_true'].nunique() < 2:
        continue
    m = calcular_metricas(valid['y_true'], valid['combined_score'])
    combined_rows.append({
        'dataset':        'combined',
        'model':          model_name,
        'family':         MODELS_REGISTRY[model_name]['family'],
        'avg_precision':  m['avg_precision'],
        'roc_auc':        m['roc_auc'],
        'precision_at_k': m['precision_at_k'],
        'f1_at_cont':     m['f1_at_cont'],
        'recall_at_k':    m['recall_at_k'],
        'recall_at_2k':   m['recall_at_2k'],
        'mean_norm_rank': m['mean_norm_rank'],
        'ndcg_at_k':      m['ndcg_at_k'],
        'time_s':         float('nan'),
    })

if combined_rows:
    ranking_df = pd.concat([ranking_df, pd.DataFrame(combined_rows)], ignore_index=True)

print("Ranking por Average Precision (com métricas PU e avaliação combinada):\n")
print(ranking_df[_cols_show].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
def _plot_curvas(ds_name, ds_results, y_test):
    """Plota curvas ROC e P×R para todos os modelos de um dataset."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    palette = sns.color_palette('tab10', n_colors=len(ds_results))

    for (model_name, res), color in zip(ds_results.items(), palette):
        scores = res['scores']
        # ROC
        fpr, tpr, _ = roc_curve(y_test, scores)
        auc_val = roc_auc_score(y_test, scores)
        axes[0].plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.3f})", color=color)
        # PR
        prec, rec, _ = precision_recall_curve(y_test, scores)
        ap_val = average_precision_score(y_test, scores)
        axes[1].plot(rec, prec, label=f"{model_name} (AP={ap_val:.3f})", color=color)

    # Baseline aleatório
    pos_rate = np.asarray(y_test).mean()
    axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random')
    axes[1].axhline(pos_rate, color='k', ls='--', lw=0.8,
                    label=f'Random (AP={pos_rate:.3f})')

    axes[0].set(title=f'Curva ROC — {ds_name}', xlabel='FPR', ylabel='TPR')
    axes[1].set(title=f'Curva PR — {ds_name}',  xlabel='Recall', ylabel='Precision')
    for ax in axes:
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.show()


def _plot_score_hist(ds_name, ds_results, y_test):
    """Histograma dos scores de anomalia por classe real."""
    n_models = len(ds_results)
    cols = min(3, n_models)
    rows = math.ceil(n_models / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows))
    axes = np.array(axes).flatten()

    for idx, (model_name, res) in enumerate(ds_results.items()):
        scores = res['scores']
        ax = axes[idx]
        for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'tomato', 'Anomalia')]:
            mask = np.asarray(y_test) == label
            ax.hist(scores[mask], bins=40, alpha=0.6, color=color,
                    label=f'{name} (n={mask.sum()})', density=True)
        ax.set_title(f'{model_name}', fontsize=10)
        ax.set_xlabel('Score'); ax.legend(fontsize=8)

    for ax in axes[n_models:]:
        ax.set_visible(False)

    plt.suptitle(f'Distribuição de Scores por Classe — {ds_name}', fontsize=12)
    plt.tight_layout()
    plt.show()


# ─────────────────────────────────────────────────────────────
# Gerar plots para cada dataset individual
# ─────────────────────────────────────────────────────────────
for ds_name, ds_results in all_results.items():
    y_test = datasets[ds_name]['y_test']
    _plot_curvas(ds_name, ds_results, y_test)
    _plot_score_hist(ds_name, ds_results, y_test)

# ─────────────────────────────────────────────────────────────
# Gerar plots para o dataset combinado
# ─────────────────────────────────────────────────────────────
if combined_dfs:
    combined_plot_results = {
        model_name: {
            'scores': df.dropna(subset=['combined_score'])['combined_score'].values
        }
        for model_name, df in combined_dfs.items()
    }
    combined_y_test = (
        next(iter(combined_dfs.values()))
        .dropna(subset=['combined_score'])['y_true'].values
    )
    _plot_curvas('combined', combined_plot_results, combined_y_test)
    _plot_score_hist('combined', combined_plot_results, combined_y_test)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Comparação: individual vs combinado (melhor dataset individual)
# ─────────────────────────────────────────────────────────────
individual_best = (
    ranking_df[ranking_df['dataset'] != 'combined']
    .groupby('model')['avg_precision'].max().reset_index()
    .rename(columns={'avg_precision': 'AP_individual_best'})
)
combined_ap = (
    ranking_df[ranking_df['dataset'] == 'combined'][['model', 'avg_precision']]
    .rename(columns={'avg_precision': 'AP_combined'})
)
comp = combined_ap.merge(individual_best, on='model')

if not comp.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(comp))
    width = 0.35
    bars1 = ax.bar(x - width/2, comp['AP_individual_best'], width, label='Melhor individual', color='steelblue')
    bars2 = ax.bar(x + width/2, comp['AP_combined'],        width, label='Combinado (max)',  color='tomato')
    ax.set_xticks(x); ax.set_xticklabels(comp['model'], rotation=20, ha='right')
    ax.set_ylabel('Average Precision')
    ax.set_title('AP: Melhor Dataset Individual vs. Score Combinado')
    ax.legend()
    ax.set_ylim(0, 1)
    for bar in list(bars1) + list(bars2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.show()

## Seção 7 — Análise de Concordância entre Modelos

Avalia quanto os modelos concordam entre si:
1. **Mapa de correlação de Spearman** dos scores (quanto os modelos ranqueiam os registros de forma similar)
2. **Contagem de consenso**: quantos modelos sinalizam cada registro como anomalia
3. **Top-20 por consenso**: registros mais suspeitos segundo o maior número de modelos

In [ ]:
from scipy.stats import spearmanr

concordance_summary = {}

for ds_name, ds_results in all_results.items():
    y_test = datasets[ds_name]['y_test']
    key_test = datasets[ds_name]['key_test']
    model_names = list(ds_results.keys())
    n = len(model_names)

    # ── 1. Correlação de Spearman entre scores ──────────────────
    scores_matrix = np.column_stack([ds_results[m]['scores'] for m in model_names])
    corr_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            corr_matrix[i, j] = spearmanr(scores_matrix[:, i], scores_matrix[:, j]).statistic

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(
        pd.DataFrame(corr_matrix, index=model_names, columns=model_names),
        annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
        ax=axes[0], square=True
    )
    axes[0].set_title(f'Correlação de Spearman (scores) — {ds_name}')

    # ── 2. Consenso: quantos modelos sinalizam cada registro ────
    preds_matrix = np.column_stack([ds_results[m]['preds'] for m in model_names])
    consensus = preds_matrix.sum(axis=1)   # número de modelos que sinalizaram 1

    consensus_df = key_test.copy()
    consensus_df['consensus_count'] = consensus
    consensus_df['is_anomaly']      = y_test.values
    concordance_summary[ds_name]    = consensus_df

    # Barplot consenso
    max_count = int(consensus.max()) + 1
    labels_order = list(range(max_count))
    counts = pd.Series(consensus).value_counts().reindex(labels_order, fill_value=0)
    counts.plot(kind='bar', ax=axes[1], color='steelblue', rot=0)
    axes[1].set_xlabel('Nº de modelos que sinalizaram anomalia')
    axes[1].set_ylabel('Nº de registros')
    axes[1].set_title(f'Histograma de Consenso — {ds_name}')
    for bar in axes[1].patches:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()

    # ── 3. Top-20 por consenso ──────────────────────────────────
    print(f"\nTop-20 registros por consenso — {ds_name}:")
    top20 = (
        consensus_df
        .sort_values('consensus_count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
    print(top20.to_string())

## Seção 8 — Exportação de Resultados

Exporta para `OUTPUT_DIR`:
- `anomaly_scores_{dataset}.parquet` — scores de todos os modelos por registro (com KEY_COLS)
- `anomaly_scores_combined.parquet` — score combinado (max) com KEY_COLS e y_true
- `model_ranking.csv` — ranking de métricas individual e combinado


In [ ]:
from datetime import datetime
sample_size_label = f"{SAMPLE_SIZE/1000}K"
execution_label = f"{sample_size_label}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(os.path.join('output', execution_label), exist_ok=True)

# ── 1. Scores por dataset (todos os modelos) ────────────────────
for ds_name, ds_results in all_results.items():
    ds = datasets[ds_name]
    export_df = ds['key_test'].copy()
    export_df['y_true'] = ds['y_test'].values
    for model_name, res in ds_results.items():
        export_df[f'score_{model_name}'] = res['scores']
        export_df[f'pred_{model_name}']  = res['preds']

    out_path = os.path.join(OUTPUT_DIR, f'anomaly_scores_{ds_name}.parquet')
    export_df.to_parquet(out_path, index=False)
    print(f"Exportado: {out_path}  ({export_df.shape})")

# ── 2. Score combinado ──────────────────────────────────────────
if combined_dfs:
    combined_ranking = ranking_df[ranking_df['dataset'] == 'combined']
    best_cm = (
        combined_ranking.sort_values('avg_precision', ascending=False).iloc[0]['model']
        if not combined_ranking.empty
        else next(iter(combined_dfs))
    )
    combined_export = combined_dfs[best_cm].copy()
    out_combined = os.path.join(OUTPUT_DIR, 'anomaly_scores_combined.parquet')
    combined_export.to_parquet(out_combined, index=False)
    print(f"Exportado: {out_combined}  ({combined_export.shape})  [melhor modelo: {best_cm}]")

# ── 3. Ranking de métricas (individual + combinado) ─────────────
out_ranking = os.path.join(OUTPUT_DIR, 'model_ranking.csv')
ranking_df.to_csv(out_ranking, index=False)
print(f"Exportado: {out_ranking}  ({ranking_df.shape})")

print("\nExportação concluída.")
print(f"\nResumo final — melhor modelo por AP (dataset estabel):")
best_row = ranking_df[ranking_df['dataset'] == 'estabel'].iloc[0]
print(f"  Modelo:           {best_row['model']}")
print(f"  Average Precision:{best_row['avg_precision']:.4f}")
print(f"  ROC-AUC:          {best_row['roc_auc']:.4f}")
print(f"  Precision@K:      {best_row['precision_at_k']:.4f}")
print(f"  F1@contamination: {best_row['f1_at_cont']:.4f}")

## Seção 9 — Grid Search de Hiperparâmetros (Opcional)

Executa busca em grade nos hiperparâmetros dos modelos cadastrados no `MODELS_REGISTRY`, utilizando o dataset configurado em `GRID_SEARCH_DATASET`.

**Fluxo:**
1. Para cada modelo em `PARAM_GRIDS`, gera todas as combinações de parâmetros via `itertools.product`
2. Instancia o modelo com `contamination=CONTAMINATION` fixo + os parâmetros da grade
3. Treina no `X_train` e avalia no `X_test` com `average_precision_score`
4. Exibe a melhor configuração por modelo e um ranking final

> **Atenção:** O grid search usa os labels do ground truth para avaliar — garanta que `LABELS_PATH` esteja configurado.  
> Esta seção é independente do loop de treinamento principal (Seção 5) e pode ser executada separadamente para explorar hiperparâmetros antes de atualizar o `MODELS_REGISTRY`.


In [ ]:
# import itertools

# # ─────────────────────────────────────────────────────────────
# # Configuração do Grid Search
# # ─────────────────────────────────────────────────────────────
# GRID_SEARCH_DATASET = 'estabel'   # dataset a usar para avaliação
# GRID_SEARCH_METRIC  = 'avg_precision'  # métrica para ordenar os resultados

# # Para cada modelo: dicionário de {parâmetro: [valores a testar]}
# # Apenas modelos listados aqui serão incluídos no grid search.
# # Os parâmetros não listados mantêm os defaults da factory do MODELS_REGISTRY.
# PARAM_GRIDS = {
#     # "IForest": {
#     #     "n_estimators": [50, 100, 200],
#     #     "max_samples":  ['auto', 256],
#     # },
#     # "HBOS": {
#     #     "n_bins": [20, 50, 100],
#     # },
#     # "LOF": {
#     #     "n_neighbors": [10, 20, 50],
#     # },
#     # "INNE": {
#     #     "n_estimators": [100, 200, 400],
#     # },
#     # "CBLOF": {
#     #     "n_clusters": [4, 8, 16],
#     #     "alpha":       [0.7, 0.9],
#     # },
#     "AutoEncoder": {
#         "hidden_neuron_list": [[32, 16, 32], [64, 32, 64], [64, 32, 32, 64]],
#         "epoch_num":         [20, 50],
#     },
# }

# # Mapeamento modelo → (classe PyOD, kwargs fixos além de contamination)
# _GS_CLASSES = {
#     "IForest":     (IForest,     {"random_state": RANDOM_STATE, "n_jobs": -1}),
#     "HBOS":        (HBOS,        {}),
#     "ECOD":        (ECOD,        {"n_jobs": -1}),
#     "COPOD":       (COPOD,       {"n_jobs": -1}),
#     "LOF":         (LOF,         {"n_jobs": -1}),
#     "INNE":        (INNE,        {"random_state": RANDOM_STATE}),
#     "CBLOF":       (CBLOF,       {"random_state": RANDOM_STATE}),
#     "AutoEncoder": (AutoEncoder, {"random_state": RANDOM_STATE, "device": _ae_device}),
# }


# def _run_grid_search(model_cls, fixed_kwargs, param_grid, X_train, X_test, y_test):
#     """
#     Itera sobre todas as combinações do param_grid e retorna DataFrame com resultados.
#     Parâmetros fixos (contamination + fixed_kwargs) são sempre incluídos.
#     """
#     rows = []
#     keys   = list(param_grid.keys())
#     values = list(param_grid.values())

#     for combo in itertools.product(*values):
#         params = dict(zip(keys, combo))
#         run_label = ", ".join(f"{k}={v}" for k, v in params.items())
#         try:
#             clf = model_cls(contamination=CONTAMINATION, **fixed_kwargs, **params)
#             t0 = time.time()
#             clf.fit(X_train)
#             scores = clf.decision_function(X_test)
#             elapsed = time.time() - t0
#             ap  = average_precision_score(y_test, scores)
#             auc = roc_auc_score(y_test, scores)
#             rows.append({
#                 "params":        run_label,
#                 "avg_precision": ap,
#                 "roc_auc":       auc,
#                 "time_s":        elapsed,
#                 **{k: str(v) for k, v in params.items()},
#             })
#         except Exception as e:
#             rows.append({
#                 "params":        run_label,
#                 "avg_precision": float("nan"),
#                 "roc_auc":       float("nan"),
#                 "time_s":        float("nan"),
#                 "error":         str(e),
#             })

#     return pd.DataFrame(rows).sort_values("avg_precision", ascending=False)


# # ─────────────────────────────────────────────────────────────
# # Execução do Grid Search
# # ─────────────────────────────────────────────────────────────
# if GRID_SEARCH_DATASET not in datasets:
#     print(f"[AVISO] Dataset '{GRID_SEARCH_DATASET}' não disponível. Execute a Seção 3 primeiro.")
# else:
#     ds_gs   = datasets[GRID_SEARCH_DATASET]
#     X_tr_gs = ds_gs['X_train']
#     X_te_gs = ds_gs['X_test']
#     y_te_gs = ds_gs['y_test'].values

#     gs_results = {}   # {model_name: DataFrame de resultados}
#     gs_best    = []   # lista de melhores por modelo (para ranking final)

#     for model_name, param_grid in PARAM_GRIDS.items():
#         if model_name not in MODELS_REGISTRY:
#             print(f"[SKIP] {model_name} não encontrado no MODELS_REGISTRY")
#             continue
#         if model_name not in _GS_CLASSES:
#             print(f"[SKIP] {model_name} sem mapeamento de classe em _GS_CLASSES")
#             continue

#         cls, fixed = _GS_CLASSES[model_name]
#         n_combos   = len(list(itertools.product(*param_grid.values())))
#         print(f"\n[Grid Search] {model_name} — {n_combos} combinações...")

#         df_gs = _run_grid_search(cls, fixed, param_grid, X_tr_gs, X_te_gs, y_te_gs)
#         gs_results[model_name] = df_gs

#         best = df_gs.iloc[0]
#         print(f"  ✓ Melhor AP={best['avg_precision']:.4f} | params: {best['params']}")
#         gs_best.append({
#             "model":         model_name,
#             "best_params":   best["params"],
#             "avg_precision": best["avg_precision"],
#             "roc_auc":       best["roc_auc"],
#             "time_s":        best["time_s"],
#         })

#     # ── Ranking final das melhores configurações ──────────────
#     if gs_best:
#         gs_best_df = pd.DataFrame(gs_best).sort_values("avg_precision", ascending=False).reset_index(drop=True)
#         print("\n" + "=" * 70)
#         print("Ranking — Melhores configurações por modelo (dataset: {})".format(GRID_SEARCH_DATASET))
#         print("=" * 70)
#         print(gs_best_df[["model", "best_params", "avg_precision", "roc_auc", "time_s"]].to_string(
#             index=False, float_format=lambda x: f"{x:.4f}"
#         ))

#         # ── Barplot comparativo ────────────────────────────────
#         fig, ax = plt.subplots(figsize=(max(6, len(gs_best_df) * 1.2), 4))
#         colors = plt.cm.tab10.colors[:len(gs_best_df)]
#         bars = ax.bar(gs_best_df["model"], gs_best_df["avg_precision"], color=colors)
#         for bar, val in zip(bars, gs_best_df["avg_precision"]):
#             ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
#                     f"{val:.4f}", ha='center', va='bottom', fontsize=9)
#         ax.set_ylabel("Average Precision (melhor config)")
#         ax.set_title(f"Grid Search — Melhor AP por Modelo ({GRID_SEARCH_DATASET})")
#         ax.set_ylim(0, min(1.0, gs_best_df["avg_precision"].max() * 1.2))
#         plt.xticks(rotation=15, ha='right')
#         plt.tight_layout()
#         plt.show()